# 05. Pipeline RAG Completo (Recuperação + Geração Fundamentada em Evidências)

**Contexto:** Tech Challenge 04 - Voice of Customer Intelligence com RAG  
**Requisitos do Desafio:**
- Demonstração completa do fluxo: Pergunta -> Transformação -> Recuperação Semântica -> Contexto -> **Geração com Modelo de Linguagem** fundamentada em evidências.
- Respostas estritamente ancoradas nas avaliações reais dos clientes (Voice of Customer).
- Prevenção ativa de alucinações e tratamento de perguntas fora do escopo ou sem dados suficientes.

**Arquitetura de Geração Plugável (interface `GeradorResposta`):**
- **`local`** (padrão) — `SintetizadorLocalVoC`: síntese por regras, **100% offline, sem chaves e sem custo**.
- **`gemini`** — `GeradorGemini`: LLM de nuvem (Google Gemini) para geração em linguagem natural, **opt-in** via variável de ambiente `GEMINI_API_KEY`, com **degradação graciosa** para o gerador local em caso de indisponibilidade (sem SDK/chave/rede).

In [7]:
import sys
import os
from IPython.display import display, Markdown
sys.path.append('../')

from src.rag_pipeline import PipelineRAG, SintetizadorLocalVoC, GeradorGemini, DEFAULT_PROMPT_TEMPLATE
from dotenv import load_dotenv, find_dotenv

def exibir_resultado_rag(titulo, resultado):
    display(Markdown(
        f"### {titulo}\n\n"
        f"> {resultado['pergunta']}\n\n"
        f"{resultado['resposta']}\n\n"
        "---"
    ))

load_dotenv(find_dotenv(usecwd=True))

DB_PATH = '../data/vector_db/'

# Leitura segura das variáveis de ambiente
GEMINI_API_KEY = os.getenv('GEMINI_API_KEY')
PROVIDER = os.getenv('GEN_PROVIDER', 'local')

# Degradação graciosa: se tentou usar gemini mas não há chave, força o fallback local
if PROVIDER == 'gemini' and not GEMINI_API_KEY:
    print("[AVISO] GEMINI_API_KEY não encontrada no .env. Alternando automaticamente para o motor 100% local.")
    PROVIDER = 'local'

# Inicialização do Pipeline
pipeline_rag = PipelineRAG(db_path=DB_PATH, provider=PROVIDER)

print(f"Pipeline RAG conectado ao ChromaDB local! Motor de geração: {type(pipeline_rag.gerador).__name__}")
print(f"Total de documentos indexados na base: {pipeline_rag.collection.count():,}")

# Diagnóstico do provider de nuvem (quando aplicável)
if isinstance(pipeline_rag.gerador, GeradorGemini):
    if pipeline_rag.gerador.disponivel:
        print(f"Gemini ativo: modelo '{pipeline_rag.gerador.model_name}'.")
    else:
        print(f"Gemini indisponível ({pipeline_rag.gerador.erro_init}). Usando fallback local automaticamente.")

Pipeline RAG conectado ao ChromaDB local! Motor de geração: SintetizadorLocalVoC
Total de documentos indexados na base: 40,668


In [8]:
import chromadb
print(chromadb.__version__)

1.5.9


## 1. Engenharia de Prompt e Diretrizes de Fidelidade (Anti-Alucinação)
Visualizamos o template de prompt que orienta a síntese analítica e impõe a recusa caso as evidências não sustentem a resposta.

In [9]:
print("=== TEMPLATE DE PROMPT E REGRAS DE GROUNDING ===")
print(DEFAULT_PROMPT_TEMPLATE)

=== TEMPLATE DE PROMPT E REGRAS DE GROUNDING ===

Você é um especialista em Inteligência de Voz do Cliente (Voice of Customer Intelligence) analisando avaliações de e-commerce.
Sua missão é responder à pergunta do gestor ESTRITAMENTE com base nas Evidências Recuperadas abaixo.

EVIDÊNCIAS RECUPERADAS DA BASE DE CONHECIMENTO:
{context}

DIRETRIZES OBRIGATÓRIAS:
1. FIDELIDADE AOS DADOS: Responda apenas com fatos sustentados pelos comentários acima. Nunca invente ou assuma informações externas.
2. RASTREABILIDADE E EVIDÊNCIAS: Cite trechos ou menções dos comentários que fundamentam cada insight apresentado.
3. TRATAMENTO DE INSUFICIÊNCIA: Se as evidências recuperadas forem insuficientes ou a pergunta estiver fora do escopo da base de clientes, responda EXATAMENTE:
   "Não possuo evidências suficientes na base de conhecimento para responder a esta pergunta."

Pergunta do Gestor: {question}

Resposta Fundamentada do Analista:



## 2. Cenário 1: Investigação de Reclamações de Entrega (Edital)
Pergunta de negócio: *"O que os clientes relatam sobre problemas relacionados à entrega?"*  
Utilizamos o **Filtro de Metadados** (`review_score <= 2`) para recuperar estritamente experiências insatisfatórias.

In [10]:
pergunta_1 = "O que os clientes relatam sobre problemas relacionados à entrega?"
resultado_1 = pipeline_rag.responder(
    pergunta=pergunta_1,
    n_results=4,
    where_filter={"review_score": {"$lte": 2}}
)

exibir_resultado_rag("Cenário 1 · Problemas com entrega", resultado_1)

PERGUNTA: O que os clientes relatam sobre problemas relacionados à entrega?

**Síntese de Voice of Customer (VoC):**
- Volume de evidências analisadas: 4 avaliações.
- Nota média ponderada dos relatos recuperados: 1.2 / 5.0 estrelas.

**Principais Problemas e Padrões de Insatisfação Identificados:**
  • Atraso/não recebimento no prazo (ID: 4baf65ba0f4dc753e9cf3d8863433da0)
  • Insatisfação geral com entrega/atendimento (ID: 12416af18a71403380ecd6e6ca53332c)
  • Insatisfação geral com entrega/atendimento (ID: 66179b1ca078cdb08c6ca238a0a4c654)
  • Insatisfação geral com entrega/atendimento (ID: c20dc8f95c7e0a17a4192984287a48ef)

**Evidências Citadas que Sustentam a Análise:**
  [1] ID `4baf65ba0f4dc753e9cf3d8863433da0` (Nota: 2): "falta de consideração com o cliente, pois ocorreu problema na entrega é em momento nenhum o problema foi tratado.duas compras, uma com prazo de entrega e , as atrasou, liguei dia reclaman"
  [2] ID `12416af18a71403380ecd6e6ca53332c` (Nota: 1): "não posso dizer 

## 3. Cenário 2: Busca por Experiências Positivas e Elogios (Edital)
Pergunta de negócio: *"Quais aspectos da experiência de compra são mais elogiados pelos clientes?"*  
Filtramos avaliações promotoras (`review_score >= 4`).

In [11]:
pergunta_2 = "Quais aspectos da experiência de compra são mais elogiados pelos clientes?"
resultado_2 = pipeline_rag.responder(
    pergunta=pergunta_2,
    n_results=4,
    where_filter={"review_score": {"$gte": 4}}
)

exibir_resultado_rag("Cenário 2 · Elogios à experiência", resultado_2)

PERGUNTA: Quais aspectos da experiência de compra são mais elogiados pelos clientes?

**Síntese de Voice of Customer (VoC):**
- Volume de evidências analisadas: 4 avaliações.
- Nota média ponderada dos relatos recuperados: 5.0 / 5.0 estrelas.

**Principais Aspectos Elogiados pelos Clientes:**
  • Recomendação positiva de compra (ID: e3def89fe0bea57dc1afb3315d4d2a1b)
  • Pontualidade e entrega antes do prazo (ID: e3def89fe0bea57dc1afb3315d4d2a1b)

**Evidências Citadas que Sustentam a Análise:**
  [1] ID `661315e416e6bc1e07ad62c9bdb44bf2` (Nota: 5): "esta pareceria entre a lannister e as demais lojas foi muito bom, pois veio a contribuir positivamente, facilitando a relação de compras entra a empresa e seus clientes."
  [2] ID `e3def89fe0bea57dc1afb3315d4d2a1b` (Nota: 5): "excelente vendedor já efetuei algumas compras sempre recebi em perfeitas condições, o produto desta compra chegou antes do prazo maximo para entrega recomendo."
  [3] ID `eebfdf7cf3787d9fac235a1faf1c4323` (Nota: 5): "s

## 4. Cenário 3: Tratamento de Perguntas sem Evidências Suficientes / Fora de Escopo
Conforme o Item 4 do edital: *"Soluções baseadas em IA Generativa podem produzir respostas aparentemente convincentes mesmo quando não existem informações suficientes para sustentá-las. Sua solução deverá considerar situações em que a pergunta esteja fora do escopo da base de conhecimento."*  

Testamos uma pergunta esdrúxula: *"Qual a cor do uniforme do entregador e qual brinde de chocolate foi enviado no pacote?"*.

In [12]:
pergunta_fora_escopo = "Qual a cor do uniforme do entregador e qual brinde de chocolate foi enviado no pacote?"
resultado_3 = pipeline_rag.responder(
    pergunta=pergunta_fora_escopo,
    n_results=3
)

exibir_resultado_rag("Cenário 3 · Pergunta fora do escopo", resultado_3)

PERGUNTA: Qual a cor do uniforme do entregador e qual brinde de chocolate foi enviado no pacote?

RESPOSTA DO SISTEMA:
Não possuo evidências suficientes na base de conhecimento para responder a esta pergunta.


## 5. Rastreabilidade das Evidências (Auditoria Completa)
O sistema permite auditar detalhadamente todos os metadados, identificadores (`review_id`) e categorias retornadas pela base vetorial.

In [30]:
linhas_auditoria = [
    f"| {i} | `{doc_id}` | {meta.get('review_score', '—')} | {meta.get('categoria') or '—'} |"
    for i, (doc_id, meta) in enumerate(zip(resultado_1['ids'], resultado_1['metadados']), 1)
]

tabela_auditoria = "\n".join(linhas_auditoria)
display(Markdown(
    "### 🔍 Auditoria das evidências da Pergunta 1\n\n"
    "| # | ID do review | Nota | Categoria |\n"
    "|---:|---|---:|---|\n"
    f"{tabela_auditoria}"
))

### 🔍 Auditoria das evidências da Pergunta 1

| # | ID do review | Nota | Categoria |
|---:|---|---:|---|
| 1 | `4baf65ba0f4dc753e9cf3d8863433da0` | 2 | papelaria |
| 2 | `12416af18a71403380ecd6e6ca53332c` | 1 | bebes |
| 3 | `66179b1ca078cdb08c6ca238a0a4c654` | 1 | cama_mesa_banho |
| 4 | `c20dc8f95c7e0a17a4192984287a48ef` | 1 | automotivo |

In [31]:
import pandas
import chromadb
import sentence_transformers
import google.genai
import dotenv

## 6. Explorações avançadas da Voz do Cliente

Nesta etapa, ampliamos as consultas ao pipeline RAG para explorar padrões de insatisfação, experiências positivas, problemas de entrega e oportunidades de melhoria na experiência dos consumidores.

In [15]:
import os
import sys

sys.path.append(os.path.abspath("../"))

print("Caminho adicionado:", os.path.abspath("../"))
print("Pasta src encontrada:", os.path.exists("../src"))

Caminho adicionado: c:\Users\Raphael\Desktop\tech-challenges\tech-challenge-04\tech-challenge-fase4
Pasta src encontrada: True


In [16]:
from src.rag_pipeline import PipelineRAG

pipeline_rag = PipelineRAG(
    db_path="../data/vector_db/",
    provider="local"
)

print("Pipeline RAG inicializado com sucesso!")

Pipeline RAG inicializado com sucesso!


In [17]:
print("client:", "client" in globals())
print("collection:", "collection" in globals())
print("pipeline_rag:", "pipeline_rag" in globals())

client: False
collection: False
pipeline_rag: True


In [29]:
perguntas_adicionais = [
    "Quais são os principais problemas relatados pelos clientes sobre a entrega?",
    "Quais aspectos da experiência de compra são mais elogiados pelos clientes?",
    "Qual a cor do uniforme do entregador e qual brinde de chocolate foi enviado no pacote?",
    "Quais avaliações sustentam problemas recorrentes de prazo?"
]

display(Markdown("## 🔎 Exploração adicional do RAG\n\nCada seção apresenta a pergunta e a síntese gerada a partir das avaliações recuperadas."))

for i, pergunta in enumerate(perguntas_adicionais, 1):
    resultado = pipeline_rag.responder(pergunta)

    exibir_resultado_rag(f"Pergunta {i:02d}", resultado)

## 🔎 Exploração adicional do RAG

Cada seção apresenta a pergunta e a síntese gerada a partir das avaliações recuperadas.

### 01 · Pergunta

> Quais são os principais problemas relatados pelos clientes sobre a entrega?

**Síntese de Voice of Customer (VoC):**
- Volume de evidências analisadas: 5 avaliações.
- Nota média ponderada dos relatos recuperados: 2.6 / 5.0 estrelas.

**Principais Problemas e Padrões de Insatisfação Identificados:**
  • Atraso/não recebimento no prazo (ID: 4baf65ba0f4dc753e9cf3d8863433da0)
  • Insatisfação geral com entrega/atendimento (ID: 12416af18a71403380ecd6e6ca53332c)

**Evidências Citadas que Sustentam a Análise:**
  [1] ID `4baf65ba0f4dc753e9cf3d8863433da0` (Nota: 2): "falta de consideração com o cliente, pois ocorreu problema na entrega é em momento nenhum o problema foi tratado.duas compras, uma com prazo de entrega e , as atrasou, liguei dia reclaman"
  [2] ID `b41fe581b62d7b9b3b0c6b16b5ccf87a` (Nota: 3): "o único problema é o envio pelos correios, pois temos que ir ao centro de distribuição pegar a encomenda. a venda e entrega foi feita por empresa parceira."
  [3] ID `12416af18a71403380ecd6e6ca53332c` (Nota: 1): "não posso dizer que não recomendo a loja, pois acredito que o problema na entrega foi com a transportadora correios . não deixaria de comprar com a loja por conta deste problema na entrega."
  [4] ID `f3a80a4c2cdd577d087958626b324741` (Nota: 3): "a terceirizacão das entregas esta comprometendo o nome da loja, pela demora na efetivacão da entrega"
  [5] ID `52ff4be3f0eb40263e7f4eca3efa639d` (Nota: 4): "único problema foi no ato da entrega onde o produto chegou no destino é voltou ao remetente por falta de comunicação."

---

### 02 · Pergunta

> Quais aspectos da experiência de compra são mais elogiados pelos clientes?

**Síntese de Voice of Customer (VoC):**
- Volume de evidências analisadas: 5 avaliações.
- Nota média ponderada dos relatos recuperados: 4.8 / 5.0 estrelas.

**Principais Aspectos Elogiados pelos Clientes:**
  • Recomendação positiva de compra (ID: e3def89fe0bea57dc1afb3315d4d2a1b)
  • Pontualidade e entrega antes do prazo (ID: e3def89fe0bea57dc1afb3315d4d2a1b)

**Evidências Citadas que Sustentam a Análise:**
  [1] ID `661315e416e6bc1e07ad62c9bdb44bf2` (Nota: 5): "esta pareceria entre a lannister e as demais lojas foi muito bom, pois veio a contribuir positivamente, facilitando a relação de compras entra a empresa e seus clientes."
  [2] ID `e3def89fe0bea57dc1afb3315d4d2a1b` (Nota: 5): "excelente vendedor já efetuei algumas compras sempre recebi em perfeitas condições, o produto desta compra chegou antes do prazo maximo para entrega recomendo."
  [3] ID `eebfdf7cf3787d9fac235a1faf1c4323` (Nota: 5): "satisfatória, é a palavra que define minha experiência de compra com essa loja. obrigado pelo atendimento. pode atuar no mercado com esse nível de compromisso."
  [4] ID `71b0927337e42e19d498bdd598e46029` (Nota: 5): "foi poditiva a experiência de compra."
  [5] ID `d756bf55cc0e9fe1d36f6fc3cc19b525` (Nota: 4): "muito bom a lojas lannister produtos muito bonito"

---

### 03 · Pergunta

> Qual a cor do uniforme do entregador e qual brinde de chocolate foi enviado no pacote?

Não possuo evidências suficientes na base de conhecimento para responder a esta pergunta.

---

### 04 · Pergunta

> Quais avaliações sustentam problemas recorrentes de prazo?

**Síntese de Voice of Customer (VoC):**
- Volume de evidências analisadas: 5 avaliações.
- Nota média ponderada dos relatos recuperados: 3.4 / 5.0 estrelas.

**Principais Problemas e Padrões de Insatisfação Identificados:**
  • Atraso/não recebimento no prazo (ID: d209b09f0dd43724c3ef350a83c85ef5)
  • Atraso/não recebimento no prazo (ID: 7015ae1a43e51e25b13198625e8ccd91)

**Principais Aspectos Elogiados pelos Clientes:**
  • Pontualidade e entrega antes do prazo (ID: 5772931f010034c3a18f55e6cde76a28)
  • Pontualidade e entrega antes do prazo (ID: e506ffd4baf43fb620898792836698d8)
  • Pontualidade e entrega antes do prazo (ID: f697051a509e926f4ba6251fe6674188)
  • Qualidade e conformidade do produto (ID: f697051a509e926f4ba6251fe6674188)

**Evidências Citadas que Sustentam a Análise:**
  [1] ID `5772931f010034c3a18f55e6cde76a28` (Nota: 5): "a princípio fiquei preocupado com o longo prazo de entrega em dias úteis, porém ele chegou um pouco antes."
  [2] ID `d209b09f0dd43724c3ef350a83c85ef5` (Nota: 2): "empresa que não entrega o produto no prazo, precisa ser avaliada são exatamente dias da compra, inclusive com o pagamento já efetuado."
  [3] ID `e506ffd4baf43fb620898792836698d8` (Nota: 4): "o produto chegou dentro do prazo estimado, mas ainda assim é um prazo muito longo."
  [4] ID `7015ae1a43e51e25b13198625e8ccd91` (Nota: 1): "não recebi nem uma satisfação pelo atraso"
  [5] ID `f697051a509e926f4ba6251fe6674188` (Nota: 5): "produto otimo, entregue antea do prazo."

---

## 6. Explorações avançadas da Voz do Cliente

Nesta etapa, ampliamos as possibilidades de análise das avaliações por meio de consultas direcionadas ao pipeline RAG. O objetivo é explorar padrões de insatisfação, experiências positivas, problemas recorrentes e oportunidades de melhoria, utilizando os metadados disponíveis para refinar a recuperação das evidências.

In [19]:
exploracoes_rag = [
    {
        "tema": "Principais problemas",
        "pergunta": "Quais são os principais problemas relatados pelos clientes?",
        "filtro": {"nota_min": 1, "nota_max": 2}
    },
    {
        "tema": "Padrões de insatisfação",
        "pergunta": "Quais situações aparecem repetidamente nos relatos de clientes insatisfeitos?",
        "filtro": {"nota_min": 1, "nota_max": 2}
    },
    {
        "tema": "Experiência de entrega",
        "pergunta": "Quais problemas relacionados à entrega são mencionados pelos consumidores?",
        "filtro": {}
    },
    {
        "tema": "Elogios dos consumidores",
        "pergunta": "Quais aspectos da experiência de compra são mais elogiados?",
        "filtro": {"nota_min": 4, "nota_max": 5}
    },
    {
        "tema": "Análise por categoria",
        "pergunta": "Quais aspectos positivos e negativos são destacados pelos clientes nesta categoria?",
        "filtro": {"categoria": "beleza_saude"}
    },
    {
        "tema": "Comparação de avaliações",
        "pergunta": "Quais diferenças aparecem entre as experiências relatadas em avaliações positivas e negativas?",
        "filtro": {}
    },
    {
        "tema": "Análise temporal",
        "pergunta": "Quais problemas e elogios aparecem nas avaliações deste período?",
        "filtro": {"ano": 2018}
    },
    {
        "tema": "Oportunidades de melhoria",
        "pergunta": "Com base nos relatos negativos, quais melhorias deveriam ser priorizadas para melhorar a experiência do cliente?",
        "filtro": {"nota_min": 1, "nota_max": 2}
    }
]

In [32]:
def descrever_filtros(filtros):
    if not filtros:
        return "Sem filtros"

    partes = []
    nota_min = filtros.get("nota_min")
    nota_max = filtros.get("nota_max")
    if nota_min is not None and nota_max is not None:
        partes.append(f"Notas {nota_min}–{nota_max}")
    elif nota_min is not None:
        partes.append(f"Notas a partir de {nota_min}")
    elif nota_max is not None:
        partes.append(f"Notas até {nota_max}")

    for chave, valor in filtros.items():
        if chave not in {"nota_min", "nota_max"}:
            rotulo = chave.replace("_", " ").capitalize()
            partes.append(f"{rotulo}: {valor}")

    return " · ".join(partes)

linhas_exploracoes = [
    f"| **{item['tema']}** | {item['pergunta']} | {descrever_filtros(item['filtro'])} |"
    for item in exploracoes_rag
]

display(Markdown(
    "## 🧭 Consultas de exploração do RAG\n\n"
    "Temas, perguntas de negócio e filtros aplicados em cada análise.\n\n"
    "| Tema | Pergunta | Filtros aplicados |\n"
    "|---|---|---|\n"
    + "\n".join(linhas_exploracoes)
))

## 🧭 Consultas de exploração do RAG

Temas, perguntas de negócio e filtros aplicados em cada análise.

| Tema | Pergunta | Filtros aplicados |
|---|---|---|
| **Principais problemas** | Quais são os principais problemas relatados pelos clientes? | Notas 1–2 |
| **Padrões de insatisfação** | Quais situações aparecem repetidamente nos relatos de clientes insatisfeitos? | Notas 1–2 |
| **Experiência de entrega** | Quais problemas relacionados à entrega são mencionados pelos consumidores? | Sem filtros |
| **Elogios dos consumidores** | Quais aspectos da experiência de compra são mais elogiados? | Notas 4–5 |
| **Análise por categoria** | Quais aspectos positivos e negativos são destacados pelos clientes nesta categoria? | Categoria: beleza_saude |
| **Comparação de avaliações** | Quais diferenças aparecem entre as experiências relatadas em avaliações positivas e negativas? | Sem filtros |
| **Análise temporal** | Quais problemas e elogios aparecem nas avaliações deste período? | Ano: 2018 |
| **Oportunidades de melhoria** | Com base nos relatos negativos, quais melhorias deveriam ser priorizadas para melhorar a experiência do cliente? | Notas 1–2 |

In [21]:
comparacao_rag = {
    "avaliacoes_negativas": {
        "pergunta": "Quais são os principais problemas e insatisfações relatados pelos consumidores?",
        "filtros": {"nota_min": 1, "nota_max": 2}
    },
    "avaliacoes_positivas": {
        "pergunta": "Quais aspectos da experiência de compra são mais elogiados pelos consumidores?",
        "filtros": {"nota_min": 4, "nota_max": 5}
    }
}

print("Consultas preparadas para comparação:")
for grupo, dados in comparacao_rag.items():
    print(f"\n{grupo.upper()}")
    print("Pergunta:", dados["pergunta"])
    print("Filtros:", dados["filtros"])

Consultas preparadas para comparação:

AVALIACOES_NEGATIVAS
Pergunta: Quais são os principais problemas e insatisfações relatados pelos consumidores?
Filtros: {'nota_min': 1, 'nota_max': 2}

AVALIACOES_POSITIVAS
Pergunta: Quais aspectos da experiência de compra são mais elogiados pelos consumidores?
Filtros: {'nota_min': 4, 'nota_max': 5}


In [22]:
resultados_exploracao = []

for item in exploracoes_rag:
    resultados_exploracao.append({
        "tema": item["tema"],
        "pergunta": item["pergunta"],
        "filtros_aplicados": item["filtro"],
        "resposta": None,
        "evidencias": None
    })

print(f"Total de consultas preparadas: {len(resultados_exploracao)}")
print("Estrutura pronta para receber respostas e evidências do RAG.")

Total de consultas preparadas: 8
Estrutura pronta para receber respostas e evidências do RAG.


## 7. Avaliação qualitativa das respostas do RAG

Para avaliar a qualidade das respostas produzidas pelo pipeline, foi estruturada uma matriz de avaliação considerando quatro critérios: aderência à pergunta, fundamentação nas evidências recuperadas, coerência com os filtros aplicados e ausência de conclusões não sustentadas pelos dados.

In [23]:
criterios_avaliacao = [
    "Aderência à pergunta",
    "Fundamentação nas evidências",
    "Coerência com os filtros",
    "Ausência de conclusões sem evidências"
]

matriz_avaliacao = []

for item in resultados_exploracao:
    matriz_avaliacao.append({
        "tema": item["tema"],
        "aderencia": None,
        "evidencias": None,
        "filtros": None,
        "confiabilidade": None,
        "observacoes": ""
    })

print("Matriz de avaliação criada!")
print("Consultas:", len(matriz_avaliacao))
print("Critérios:", criterios_avaliacao)

Matriz de avaliação criada!
Consultas: 8
Critérios: ['Aderência à pergunta', 'Fundamentação nas evidências', 'Coerência com os filtros', 'Ausência de conclusões sem evidências']


In [24]:
import pandas as pd

df_avaliacao = pd.DataFrame([
    {
        "Tema": item["tema"],
        "Aderência à pergunta": "",
        "Fundamentação nas evidências": "",
        "Coerência com os filtros": "",
        "Ausência de conclusões sem evidências": "",
        "Observações": ""
    }
    for item in resultados_exploracao
])

display(df_avaliacao)

,Tema,Aderência à pergunta,Fundamentação nas evidências,Coerência com os filtros,Ausência de conclusões sem evidências,Observações
0,Principais problemas,,,,,
1,Padrões de insatisfação,,,,,
2,Experiência de entrega,,,,,
3,Elogios dos consumidores,,,,,
4,Análise por categoria,,,,,
5,Comparação de avaliações,,,,,
6,Análise temporal,,,,,
7,Oportunidades de melhoria,,,,,


In [25]:
def calcular_resultado_avaliacao(df):
    criterios = [
        "Aderência à pergunta",
        "Fundamentação nas evidências",
        "Coerência com os filtros",
        "Ausência de conclusões sem evidências"
    ]

    notas = df[criterios].apply(pd.to_numeric, errors="coerce")

    df["Média geral"] = notas.mean(axis=1).round(2)

    return df

print("Função de avaliação criada.")

Função de avaliação criada.


In [26]:
def classificar_qualidade(media):
    if pd.isna(media):
        return "Não avaliada"
    elif media >= 4.5:
        return "Excelente"
    elif media >= 3.5:
        return "Boa"
    elif media >= 2.5:
        return "Regular"
    else:
        return "Precisa melhorar"

print("Classificação automática criada!")

Classificação automática criada!
